# DeBERTa-v3 multi-task toxicity fine-tune -- Kaggle Notebook

Runs the exact same `src/` code developed and unit-tested locally
(this notebook just re-materializes those files here via `%%writefile`,
generated by `scripts/gen_kaggle_notebook.py` -- do not hand-edit the
code cells below, regenerate instead so this never drifts from what was
actually tested).

**Before running:**
1. Settings (right panel) -> Accelerator -> **GPU T4 x2** or **P100**.
2. Add data -> search **"Jigsaw Unintended Bias in Toxicity Classification"**
   -> Add. This mounts the competition files under
   `/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/`.
3. Settings -> Internet -> **On** (needed to download `deberta-v3-base`
   weights from Hugging Face on first run).
4. Session length: full-data training is ~3.5-4.5h/epoch x 2 epochs on a
   T4 -- comfortably inside a 9h session, but checkpoints are written
   every 2000 steps to `/kaggle/working/models/` in case of a disconnect.

**To run the ablations referenced in the README:** edit `config.yaml`
below (the `%%writefile config.yaml` cell) -- flip `use_multitask_heads`
and/or `weighting_scheme`, re-run that cell + the training cell, save a
new notebook version per configuration so each run's output is preserved.


## 1. Install extra dependencies

`torch`/`transformers` ship preinstalled on Kaggle GPU notebooks. The preinstalled torch build (observed: 2.10.0+cu128) has dropped kernel support for Pascal-generation GPUs (sm_60, e.g. the P100 Kaggle sometimes assigns) -- confirmed the hard way: `CUDA error: no kernel image is available for execution on the device`, right after `torch.zeros(..., device='cuda')`. `machine_shape` in kernel-metadata.json did not reliably control which GPU an API-pushed kernel gets, so instead of gambling on GPU assignment, we detect the actual device's compute capability and reinstall a torch/cu121 build (still spans Pascal through Hopper) only if the preinstalled one can't run a kernel on this GPU.

In [ ]:
!pip install -q sentencepiece pyyaml


In [ ]:
import subprocess
import torch

def cuda_actually_works() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        x = torch.zeros(4, device='cuda')
        _ = x + 1
        torch.cuda.synchronize()
        return True
    except Exception as e:
        print(f'CUDA smoke op failed on preinstalled torch: {e}')
        return False

print('torch', torch.__version__, 'cuda build', torch.version.cuda)
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

if not cuda_actually_works():
    print('Reinstalling a broader-compatibility torch build (cu121)...')
    subprocess.run(
        ['pip', 'install', '-q', '--force-reinstall',
         'torch==2.4.1', '--index-url', 'https://download.pytorch.org/whl/cu121'],
        check=True,
    )
    # We don't use torchvision/torchaudio anywhere -- but they stay
    # pinned to the OLD torch build's version after the line above,
    # which breaks their compiled ops (torchvision::nms) against the
    # new torch, which in turn breaks transformers' DebertaV2Model
    # import (it probes torchvision at import time). Simplest fix:
    # remove what we don't need instead of chasing matched versions.
    subprocess.run(['pip', 'uninstall', '-y', '-q', 'torchvision', 'torchaudio'], check=True)
    # Verify via a FRESH subprocess, not this kernel's already-imported
    # torch module -- Python cannot cleanly reload a C-extension
    # module in-process, but train.py runs as its own subprocess
    # later anyway and will import the newly installed package fine
    # regardless of what this cell's `torch` object thinks.
    check = subprocess.run(
        ['python', '-c',
         "import torch; x=torch.zeros(4, device='cuda'); y=x+1; torch.cuda.synchronize(); "
         "print('torch', torch.__version__, 'cuda build', torch.version.cuda); print('CUDA OK:', y)"],
    )
    assert check.returncode == 0, 'CUDA still broken after torch reinstall -- needs manual investigation.'
else:
    print('Preinstalled torch already works on this GPU -- no reinstall needed.')


In [ ]:
import os
# Sanity check the data mount before spending any GPU time on it --
# API-pushed kernels mount competition data at
# /kaggle/input/competitions/<slug>/, NOT /kaggle/input/<slug>/ like
# the browser 'Add Data' flow does. Confirmed the hard way once.
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))


## 2. Materialize the project's src/ modules

Identical content to the local repo's `src/metrics.py`, `src/data.py`, `src/model.py`, `src/train.py` -- see that repo for the documented rationale behind each design choice (bias metric derivation, weighting scheme, multi-task heads).

In [ ]:
%%writefile metrics.py
"""
Jigsaw "Unintended Bias in Toxicity Classification" evaluation metric.

Implemented from the competition's written definition (not copied from a
public kernel) so the reasoning is auditable end to end.

Background
----------
A single overall AUC hides subgroup-level bias: a model can separate toxic
from non-toxic comments very well in aggregate while systematically mis-
scoring comments that merely *mention* a protected identity. The competition
metric decomposes AUC into three views per identity subgroup, then combines
across subgroups with a generalized mean that punishes the worst subgroup
much harder than a plain average would.

Three subgroup-conditioned AUCs
--------------------------------
For a given identity subgroup S (e.g. "muslim"), let:
    - subgroup    = rows where the identity is mentioned (indicator == 1)
    - background  = rows where the identity is NOT mentioned

1. Subgroup AUC
   AUC computed using ONLY rows in `subgroup`. Answers: "within comments
   that mention this identity, can the model separate toxic from
   non-toxic?" A low value means the model is generally confused about
   this subgroup's comments (not specifically a false-positive/negative
   problem, just noisy).

2. BPSN AUC (Background Positive, Subgroup Negative)
   AUC computed on:
       - subgroup   rows that are NON-toxic  (labelled 0 / negative)
       - background rows that ARE   toxic    (labelled 1 / positive)
   This isolates the false-positive failure mode: benign subgroup mentions
   scoring as high as, or higher than, genuine toxicity elsewhere. This is
   the "I am a gay man" problem -- a low BPSN AUC means the identity term
   itself is acting as a toxicity signal.

3. BNSP AUC (Background Negative, Subgroup Positive)
   The mirror image:
       - subgroup   rows that ARE   toxic    (labelled 1 / positive)
       - background rows that are NON-toxic  (labelled 0 / negative)
   A low value means real attacks on the subgroup are being under-flagged
   relative to background toxicity.

Generalized power mean (p = -5)
--------------------------------
Given the per-subgroup scores for one of the three metrics above, combine
them with:

    M_p(x_1, ..., x_n) = ( mean(x_i ** p) ) ** (1 / p)

At p = -5 this behaves like a *soft minimum*: it is dominated by whichever
subgroup scores worst, so you cannot average your way to a good number by
doing well on eight subgroups and badly on one.

Final competition score
------------------------
    score = 0.25 * overall_AUC
           + 0.25 * power_mean(subgroup_AUC over all subgroups)
           + 0.25 * power_mean(BPSN_AUC over all subgroups)
           + 0.25 * power_mean(BNSP_AUC over all subgroups)
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# The nine subgroups the competition actually scores on (large enough
# sample size in the identity-annotated subset to be statistically
# meaningful). Other identity columns exist in the raw data (hindu,
# buddhist, latino, ...) but are too sparse to score reliably -- keep them
# for error analysis, not for the headline metric.
IDENTITY_COLUMNS = [
    "male",
    "female",
    "homosexual_gay_or_lesbian",
    "christian",
    "jewish",
    "muslim",
    "black",
    "white",
    "psychiatric_or_mental_illness",
]

# Six auxiliary toxicity subtypes used for the multi-task head, and useful
# on their own for error-mode analysis.
AUX_TOXICITY_COLUMNS = [
    "severe_toxicity",
    "obscene",
    "threat",
    "insult",
    "identity_attack",
    "sexual_explicit",
]

POWER_MEAN_P = -5


def power_mean(values, p: float = POWER_MEAN_P) -> float:
    """Generalized (Hölder) power mean, robust to zeros/negatives in `values`.

    For p < 0 this is undefined if any value is exactly 0 (division by
    zero inside the mean). AUCs are essentially never exactly 0 in
    practice; if it happens we clip to a tiny epsilon rather than raising,
    so one degenerate subgroup doesn't crash the whole evaluation.
    """
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return float("nan")
    eps = 1e-12
    values = np.clip(values, eps, None)
    return float(np.mean(values**p) ** (1.0 / p))


def _safe_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """roc_auc_score that returns NaN instead of raising when a split has
    only one class present (can happen on very small/sparse subgroups)."""
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_pred))


def subgroup_auc(df: pd.DataFrame, subgroup_col: str, label_col: str, pred_col: str) -> float:
    mask = df[subgroup_col] >= 0.5
    sub = df.loc[mask]
    return _safe_auc(sub[label_col].values, sub[pred_col].values)


def bpsn_auc(df: pd.DataFrame, subgroup_col: str, label_col: str, pred_col: str) -> float:
    subgroup_negative = df[(df[subgroup_col] >= 0.5) & (df[label_col] == 0)]
    background_positive = df[(df[subgroup_col] < 0.5) & (df[label_col] == 1)]
    combined = pd.concat([subgroup_negative, background_positive])
    return _safe_auc(combined[label_col].values, combined[pred_col].values)


def bnsp_auc(df: pd.DataFrame, subgroup_col: str, label_col: str, pred_col: str) -> float:
    subgroup_positive = df[(df[subgroup_col] >= 0.5) & (df[label_col] == 1)]
    background_negative = df[(df[subgroup_col] < 0.5) & (df[label_col] == 0)]
    combined = pd.concat([subgroup_positive, background_negative])
    return _safe_auc(combined[label_col].values, combined[pred_col].values)


@dataclass
class BiasMetricResult:
    per_subgroup: pd.DataFrame
    overall_auc: float
    subgroup_auc_power_mean: float
    bpsn_auc_power_mean: float
    bnsp_auc_power_mean: float
    final_score: float
    identity_columns: list = field(default_factory=list)

    def summary(self) -> pd.Series:
        return pd.Series(
            {
                "overall_auc": self.overall_auc,
                "subgroup_auc_power_mean": self.subgroup_auc_power_mean,
                "bpsn_auc_power_mean": self.bpsn_auc_power_mean,
                "bnsp_auc_power_mean": self.bnsp_auc_power_mean,
                "final_score": self.final_score,
            }
        )

    def worst_subgroups(self, metric: str = "bpsn_auc", n: int = 3) -> pd.DataFrame:
        return self.per_subgroup.sort_values(metric).head(n)


def compute_bias_metrics(
    df: pd.DataFrame,
    label_col: str = "target",
    pred_col: str = "prediction",
    identity_cols: list | None = None,
    label_threshold: float = 0.5,
    power_p: float = POWER_MEAN_P,
) -> BiasMetricResult:
    """Compute the full Jigsaw bias-metric report.

    Parameters
    ----------
    df : DataFrame containing, at minimum, `label_col`, `pred_col`, and one
        column per identity in `identity_cols`. `label_col` may be the raw
        soft fraction (e.g. 0.0-1.0 `target`) -- it is binarized here at
        `label_threshold`. Identity columns are expected in the same
        fractional form (proportion of raters who tagged that identity);
        rows with NaN identity values are treated as "identity not
        mentioned" (< 0.5) which matches how the competition scores the
        un-annotated 1.35M rows: they simply never enter any subgroup mask.
    label_col : soft or hard toxicity label column.
    pred_col : model score column (higher = more toxic).
    identity_cols : defaults to the 9 competition subgroups.
    label_threshold : binarization threshold for `label_col` (competition
        uses 0.5).
    power_p : exponent for the generalized mean (competition uses -5).

    Returns
    -------
    BiasMetricResult
    """
    if identity_cols is None:
        identity_cols = IDENTITY_COLUMNS

    missing = [c for c in identity_cols + [label_col, pred_col] if c not in df.columns]
    if missing:
        raise KeyError(f"compute_bias_metrics: missing columns {missing}")

    work = df.copy()
    # Binarize the label. NaN identity columns are filled with 0 so rows
    # without annotations simply never qualify for any subgroup mask
    # (they still count in "background" for BPSN/BNSP, which is correct:
    # the un-annotated 1.35M rows *are* the background population).
    work[label_col] = (work[label_col] >= label_threshold).astype(int)
    for c in identity_cols:
        work[c] = work[c].fillna(0.0)

    rows = []
    for identity in identity_cols:
        rows.append(
            {
                "subgroup": identity,
                "subgroup_size": int((work[identity] >= 0.5).sum()),
                "subgroup_auc": subgroup_auc(work, identity, label_col, pred_col),
                "bpsn_auc": bpsn_auc(work, identity, label_col, pred_col),
                "bnsp_auc": bnsp_auc(work, identity, label_col, pred_col),
            }
        )
    per_subgroup = pd.DataFrame(rows).set_index("subgroup")

    overall = _safe_auc(work[label_col].values, work[pred_col].values)

    sg_pm = power_mean(per_subgroup["subgroup_auc"].dropna().values, power_p)
    bpsn_pm = power_mean(per_subgroup["bpsn_auc"].dropna().values, power_p)
    bnsp_pm = power_mean(per_subgroup["bnsp_auc"].dropna().values, power_p)

    final_score = 0.25 * overall + 0.25 * sg_pm + 0.25 * bpsn_pm + 0.25 * bnsp_pm

    return BiasMetricResult(
        per_subgroup=per_subgroup,
        overall_auc=overall,
        subgroup_auc_power_mean=sg_pm,
        bpsn_auc_power_mean=bpsn_pm,
        bnsp_auc_power_mean=bnsp_pm,
        final_score=final_score,
        identity_columns=identity_cols,
    )


if __name__ == "__main__":
    # Self-test with synthetic data: no need for the real 1.8M-row dataset
    # to verify the metric implementation is wired correctly. We construct
    # a deliberately biased toy model (identity term -> inflated score) and
    # check that BPSN correctly tanks for the biased subgroup while a
    # matched "clean" subgroup scores well.
    rng = np.random.default_rng(0)
    n = 4000
    toxic = rng.integers(0, 2, size=n).astype(float)  # hard 0/1 "target"
    mentions_biased = rng.integers(0, 2, size=n).astype(float)  # e.g. "muslim"
    mentions_clean = rng.integers(0, 2, size=n).astype(float)  # e.g. "male"

    base_score = toxic * 0.35 + rng.normal(0, 0.15, n)
    # Biased model: mentioning the identity inflates the score regardless
    # of true toxicity -> should hurt BPSN for `mentions_biased`, since
    # non-toxic comments that merely mention it now compete on score with
    # genuinely toxic background comments.
    biased_pred = base_score + mentions_biased * 0.35
    clean_pred = base_score.copy()

    toy = pd.DataFrame(
        {
            "target": toxic,
            "muslim": mentions_biased,
            "male": mentions_clean,
            "prediction_biased": np.clip(biased_pred, 0, 1),
            "prediction_clean": np.clip(clean_pred, 0, 1),
        }
    )
    # fill the other 7 required identity columns with 0 (not mentioned)
    for c in IDENTITY_COLUMNS:
        if c not in toy.columns:
            toy[c] = 0.0

    for pred_col in ["prediction_biased", "prediction_clean"]:
        result = compute_bias_metrics(toy, pred_col=pred_col)
        print(f"\n=== {pred_col} ===")
        print(result.per_subgroup.loc[["muslim", "male"]])
        print(result.summary())

    print(
        "\nExpected: prediction_biased has a visibly lower bpsn_auc on "
        "'muslim' than on 'male' -- the identity term itself is inflating "
        "scores, so benign 'muslim' comments compete with genuinely toxic "
        "background comments and BPSN drops. prediction_clean carries the "
        "same real toxic-vs-not signal without the identity inflation, so "
        "it scores well and equally on both subgroups (no bias gap)."
    )


In [ ]:
%%writefile data.py
"""
Data loading, sample weighting, and splitting for the Jigsaw Unintended
Bias dataset.

Sample-weighting scheme
------------------------
We do NOT use SMOTE or naive class oversampling here -- see the README for
why (short version: SMOTE is meaningless on transformer text embeddings,
and naively oversampling the toxic class amplifies whatever identity
correlations already exist in that class, making the bias problem worse,
not better).

Instead we use the metric-derived weighting scheme published with the
competition. It is derived directly from the evaluation metric (see
`metrics.py`), not from the class imbalance in isolation:

    w  = 0.25                                          # base weight
    w += 0.25 * [comment mentions any of the 9 subgroups]
    w += 0.25 * [toxic AND mentions no subgroup]         # background positive
    w += 0.25 * [non-toxic AND mentions a subgroup]      # subgroup negative

The last term is the important one: it upweights exactly the examples
whose misclassification tanks BPSN AUC (benign comments that mention an
identity). The third term does the mirror job for BNSP (real attacks that
don't happen to mention a tracked identity, which would otherwise be
drowned out once we start upweighting subgroup mentions). Weights are
un-normalized on purpose -- what matters for a weighted loss is their
*relative* scale, and BCE loss functions accept per-example weights
directly.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from metrics import AUX_TOXICITY_COLUMNS, IDENTITY_COLUMNS

DEFAULT_TOXIC_THRESHOLD = 0.5
DEFAULT_IDENTITY_THRESHOLD = 0.5


def load_raw(path: str | Path, usecols: list[str] | None = None) -> pd.DataFrame:
    """Load the raw Jigsaw train.csv.

    `usecols=None` loads every column (comment_text, target, the 9 scored
    identities, ~15 other identity columns, the 6 aux subtypes, and some
    metadata columns like rating counts). Pass an explicit list to save
    memory during development.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Download train.csv from the Kaggle "
            "'jigsaw-unintended-bias-in-toxicity-classification' "
            "competition and place it under data/raw/."
        )
    return pd.read_csv(path, usecols=usecols)


def mentions_any_subgroup(
    df: pd.DataFrame,
    identity_cols: list[str] = IDENTITY_COLUMNS,
    threshold: float = DEFAULT_IDENTITY_THRESHOLD,
) -> pd.Series:
    """Boolean: does this row have >=1 of the 9 scored identities annotated
    at >= threshold? NaN identity values (the 1.35M un-annotated rows)
    count as "not mentioned", which is the correct interpretation -- we
    have no annotator evidence either way, so we do not claim membership.
    """
    present = df[identity_cols].fillna(0.0) >= threshold
    return present.any(axis=1)


def compute_sample_weights(
    df: pd.DataFrame,
    target_col: str = "target",
    identity_cols: list[str] = IDENTITY_COLUMNS,
    toxic_threshold: float = DEFAULT_TOXIC_THRESHOLD,
    identity_threshold: float = DEFAULT_IDENTITY_THRESHOLD,
) -> pd.Series:
    """Metric-derived sample weights. See module docstring for the
    derivation. Returns a float Series aligned to df.index, range [0.25, 1.0].
    """
    toxic = df[target_col] >= toxic_threshold
    has_identity = mentions_any_subgroup(df, identity_cols, identity_threshold)

    w = pd.Series(0.25, index=df.index, dtype="float64")
    w += 0.25 * has_identity.astype("float64")
    w += 0.25 * (toxic & ~has_identity).astype("float64")
    w += 0.25 * (~toxic & has_identity).astype("float64")
    return w


def compute_inverse_frequency_weights(df: pd.DataFrame, target_col: str = "target", toxic_threshold: float = DEFAULT_TOXIC_THRESHOLD) -> pd.Series:
    """Plain class-balance baseline for the ablation table (NOT the
    scheme we ship): weight = 1 / P(class). Included only so the README's
    "no weighting vs inverse-frequency vs metric-derived" comparison has a
    fair, standard second row.
    """
    toxic = df[target_col] >= toxic_threshold
    p_toxic = toxic.mean()
    p_nontoxic = 1 - p_toxic
    w = pd.Series(np.where(toxic, 1.0 / p_toxic, 1.0 / p_nontoxic), index=df.index)
    return w


def train_val_split(
    df: pd.DataFrame,
    val_size: float = 0.05,
    seed: int = 42,
    target_col: str = "target",
    toxic_threshold: float = DEFAULT_TOXIC_THRESHOLD,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Stratified split on (toxic label x has-any-identity) so the rare
    "toxic and mentions an identity" stratum -- the one BPSN/BNSP care
    about most -- doesn't get unlucky in a plain random split.
    """
    from sklearn.model_selection import train_test_split

    strata = (
        (df[target_col] >= toxic_threshold).astype(int).astype(str)
        + "_"
        + mentions_any_subgroup(df).astype(int).astype(str)
    )
    train_df, val_df = train_test_split(
        df, test_size=val_size, random_state=seed, stratify=strata
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def stratified_subsample(
    df: pd.DataFrame,
    n: int = 200_000,
    seed: int = 42,
    target_col: str = "target",
    toxic_threshold: float = DEFAULT_TOXIC_THRESHOLD,
    min_per_subgroup: int = 500,
) -> pd.DataFrame:
    """Fast-iteration dev subsample.

    Plain proportional stratified sampling on (toxic x has-identity) would
    still leave individual sparse subgroups (e.g. `muslim`) with very few
    rows once you're down to 200k, which makes the bias metrics on the dev
    subsample noisy to the point of being useless for iteration. So: do a
    proportional stratified sample first, then top up each of the 9 scored
    subgroups to at least `min_per_subgroup` rows by adding back randomly
    sampled rows that mention it (dropping duplicates). This keeps the
    toxic base rate roughly intact while guaranteeing every subgroup table
    is estimated on a meaningful sample during development.
    """
    from sklearn.model_selection import train_test_split

    n = min(n, len(df))
    strata = (
        (df[target_col] >= toxic_threshold).astype(int).astype(str)
        + "_"
        + mentions_any_subgroup(df).astype(int).astype(str)
    )
    sample, _ = train_test_split(
        df, train_size=n, random_state=seed, stratify=strata
    )
    sample = sample.copy()

    rng = np.random.default_rng(seed)
    topped_up = [sample]
    for identity in IDENTITY_COLUMNS:
        current_count = int((sample[identity].fillna(0.0) >= DEFAULT_IDENTITY_THRESHOLD).sum())
        if current_count >= min_per_subgroup:
            continue
        pool = df[df[identity].fillna(0.0) >= DEFAULT_IDENTITY_THRESHOLD]
        pool = pool[~pool.index.isin(sample.index)]
        need = min_per_subgroup - current_count
        if len(pool) == 0:
            continue
        extra = pool.sample(n=min(need, len(pool)), random_state=int(rng.integers(0, 1_000_000)))
        topped_up.append(extra)

    result = pd.concat(topped_up).drop_duplicates(subset=[df.columns[0]] if len(df.columns) else None)
    # drop_duplicates on the id column if present, else on the whole row
    if "id" in df.columns:
        result = result.drop_duplicates(subset="id")
    return result.reset_index(drop=True)


def prepare_training_frame(
    df: pd.DataFrame,
    weighting: str = "metric_derived",
) -> pd.DataFrame:
    """Attach sample_weight column and binarized aux-subtype targets
    (filled 0 for rows missing annotations, matching how the competition's
    training kernels handle it -- the aux heads only get meaningful
    gradient on rows that have subtype annotations, which is fine since
    they're an auxiliary regularizer, not the scored output).

    `weighting`: one of {"none", "inverse_frequency", "metric_derived"}.
    """
    out = df.copy()
    if weighting == "none":
        out["sample_weight"] = 1.0
    elif weighting == "inverse_frequency":
        out["sample_weight"] = compute_inverse_frequency_weights(out)
    elif weighting == "metric_derived":
        out["sample_weight"] = compute_sample_weights(out)
    else:
        raise ValueError(f"unknown weighting scheme: {weighting}")

    for c in AUX_TOXICITY_COLUMNS:
        if c in out.columns:
            out[c] = out[c].fillna(0.0)
    return out


if __name__ == "__main__":
    # Smoke test on synthetic data shaped like the real schema, so this
    # module is verifiable before the real 1.8M-row CSV is downloaded.
    rng = np.random.default_rng(0)
    n = 20_000
    toy = pd.DataFrame({"id": np.arange(n), "target": rng.random(n)})
    for c in IDENTITY_COLUMNS:
        # ~70% NaN (unannotated), else a fractional rater score
        vals = np.where(rng.random(n) < 0.7, np.nan, rng.random(n))
        toy[c] = vals
    for c in AUX_TOXICITY_COLUMNS:
        toy[c] = rng.random(n) * (toy["target"] >= 0.5)

    weighted = prepare_training_frame(toy, weighting="metric_derived")
    print(weighted["sample_weight"].describe())
    print("\nweight value counts (should be a handful of discrete levels):")
    print(weighted["sample_weight"].value_counts().sort_index())

    train_df, val_df = train_val_split(toy, val_size=0.1)
    print(f"\ntrain={len(train_df)} val={len(val_df)}")

    sub = stratified_subsample(toy, n=5000, min_per_subgroup=200)
    print(f"\nsubsample size={len(sub)}")
    for c in IDENTITY_COLUMNS:
        print(c, int((sub[c].fillna(0.0) >= 0.5).sum()))


In [ ]:
%%writefile model.py
"""
DeBERTa-v3 encoder with a multi-task classification head.

Architecture rationale (see README "Stage 2" for the full argument): a
single toxicity head can satisfy its loss by learning "identity term
present -> toxic", because that shortcut correlates with the training
label often enough to lower the loss. Forcing the model to also predict
the 6 auxiliary subtypes (severe_toxicity, obscene, threat, insult,
identity_attack, sexual_explicit) makes that shortcut actively harmful:
a comment that just mentions an identity with no insult/threat/attack
content now gets a loss penalty on the aux heads if the model scores it
toxic-shaped, which pushes the encoder to separate "identity mentioned"
from "identity attacked". This is a regularizer that targets the exact
failure mode BPSN measures.

`use_multitask_heads=False` collapses this to a single-head baseline, so
the same class can produce both arms of the ablation.
"""

from __future__ import annotations

import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel


class MultiTaskToxicityModel(nn.Module):
    def __init__(
        self,
        model_name: str = "microsoft/deberta-v3-base",
        n_aux_labels: int = 6,
        use_multitask_heads: bool = True,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.use_multitask_heads = use_multitask_heads
        self.config = AutoConfig.from_pretrained(model_name)
        # low_cpu_mem_usage=False: force the traditional (instantiate-then-
        # load_state_dict) weight-loading path instead of accelerate's
        # meta-device "materialize per-parameter" path, which transformers
        # enables by default whenever accelerate is importable. That path
        # is meant for models too large to instantiate twice in RAM -- not
        # a concern at 184M params -- and it silently crashed mid-load
        # (no Python traceback, just a dead subprocess) on a Kaggle T4
        # GPU node with DeBERTa-v2's custom disentangled-attention modules.
        # use_safetensors=True: on a torch build old enough to still support
        # Pascal-generation GPUs (see the low_cpu_mem_usage note above --
        # Kaggle can hand out a P100), transformers refuses to torch.load()
        # a raw .bin checkpoint at all (a security gate that requires
        # torch>=2.6) and raises rather than falling back on its own.
        # Safetensors loading has no such restriction and Microsoft's
        # official checkpoint ships both formats.
        #
        # torch_dtype=torch.float32: transformers can default to whatever
        # dtype the checkpoint's config declares (some hub configs say
        # float16) unless told otherwise. That silently breaks the
        # autocast+GradScaler mixed-precision pattern in train.py, which
        # requires FP32 master parameters -- confirmed the hard way via
        # `ValueError: Attempting to unscale FP16 gradients` the moment
        # backward() ran. autocast handles the fp16 compute internally;
        # the stored parameters must stay fp32.
        self.encoder = AutoModel.from_pretrained(
            model_name,
            config=self.config,
            low_cpu_mem_usage=False,
            use_safetensors=True,
            torch_dtype=torch.float32,
        )
        hidden = self.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.main_head = nn.Linear(hidden, 1)  # target (soft toxicity)
        if use_multitask_heads:
            self.aux_head = nn.Linear(hidden, n_aux_labels)
        else:
            self.aux_head = None

    def _pool(self, last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        # Mean pooling over non-padding tokens. Simpler and generally more
        # robust than the raw [CLS]/first-token vector for this encoder,
        # and cheap.
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**encoder_kwargs)
        pooled = self._pool(out.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)

        main_logit = self.main_head(pooled).squeeze(-1)  # (batch,)
        aux_logits = self.aux_head(pooled) if self.aux_head is not None else None  # (batch, n_aux)
        return main_logit, aux_logits


def multitask_loss(
    main_logit: torch.Tensor,
    main_target: torch.Tensor,
    sample_weight: torch.Tensor,
    aux_logits: torch.Tensor | None = None,
    aux_targets: torch.Tensor | None = None,
    aux_loss_weight: float = 0.25,
) -> torch.Tensor:
    """Weighted BCE on the main soft-label target, plus (optionally) an
    unweighted BCE on the 6 aux subtype heads. Main target and predictions
    are kept as soft probabilities (BCEWithLogits against the raw fraction
    in [0,1], not a binarized 0/1) -- this is the "free signal" the spec
    calls out: training against the rater-agreement fraction directly
    rather than throwing it away at the 0.5 threshold.
    """
    main_loss_per_example = nn.functional.binary_cross_entropy_with_logits(
        main_logit, main_target, reduction="none"
    )
    main_loss = (main_loss_per_example * sample_weight).sum() / sample_weight.sum().clamp(min=1e-6)

    if aux_logits is None or aux_targets is None:
        return main_loss

    aux_loss = nn.functional.binary_cross_entropy_with_logits(
        aux_logits, aux_targets, reduction="mean"
    )
    return main_loss + aux_loss_weight * aux_loss


if __name__ == "__main__":
    # Structural smoke test with a tiny model so this runs on CPU in
    # seconds without downloading deberta-v3-base. Swap model_name for the
    # real one when running with GPU access (Kaggle/Colab).
    tiny_model_name = "hf-internal-testing/tiny-random-DebertaV2Model"
    try:
        model = MultiTaskToxicityModel(model_name=tiny_model_name, use_multitask_heads=True)
    except Exception as e:
        print(f"Skipping model.py smoke test (no network access to fetch a tiny HF model): {e}")
    else:
        batch, seq_len = 4, 16
        input_ids = torch.randint(0, model.config.vocab_size, (batch, seq_len))
        attention_mask = torch.ones(batch, seq_len, dtype=torch.long)
        main_logit, aux_logits = model(input_ids, attention_mask)
        print("main_logit shape:", main_logit.shape)
        print("aux_logits shape:", None if aux_logits is None else aux_logits.shape)

        target = torch.rand(batch)
        weight = torch.ones(batch)
        aux_targets = torch.rand(batch, 6)
        loss = multitask_loss(main_logit, target, weight, aux_logits, aux_targets)
        print("loss:", loss.item())


In [ ]:
%%writefile train.py
"""
Fine-tuning driver for the DeBERTa-v3 multi-task toxicity model.

Meant to run where a GPU is available (Kaggle/Colab) — see README for
timing (roughly 3.5-4.5h/epoch on a T4 at max_len=220, batch=32, over the
full 1.8M rows). Locally (CPU-only) use `--sample` for a structural
smoke test only; do not expect a real result from a CPU run.

Everything hyperparameter-shaped is read from config/config.yaml so the
weighting-scheme ablation (none / inverse_frequency / metric_derived) and
the multi-task-head ablation (on/off) are just config flips, not code
changes — that's what makes the ablation table in the README reproducible
from `make train CONFIG=...` rather than from hand-edited notebook cells.

Usage:
    python src/train.py --config config/config.yaml
    python src/train.py --config config/config.yaml --sample 2000  # smoke test
"""

from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from data import load_raw, prepare_training_frame, train_val_split
from metrics import AUX_TOXICITY_COLUMNS, IDENTITY_COLUMNS, compute_bias_metrics
from model import MultiTaskToxicityModel, multitask_loss

ROOT = Path(__file__).resolve().parent.parent


class ToxicityDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, aux_cols: list[str]):
        self.texts = df["comment_text"].fillna("").tolist()
        self.targets = df["target"].astype("float32").values
        self.weights = df["sample_weight"].astype("float32").values
        self.aux = df[aux_cols].astype("float32").values if all(c in df.columns for c in aux_cols) else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "target": torch.tensor(self.targets[idx], dtype=torch.float32),
            "sample_weight": torch.tensor(self.weights[idx], dtype=torch.float32),
        }
        if self.aux is not None:
            item["aux_target"] = torch.tensor(self.aux[idx], dtype=torch.float32)
        return item


def load_config(path: str) -> dict:
    with open(path) as f:
        return yaml.safe_load(f)


@torch.no_grad()
def run_inference(model, loader, device) -> np.ndarray:
    model.eval()
    all_logits = []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        main_logit, _ = model(input_ids, attention_mask)
        all_logits.append(main_logit.detach().cpu().numpy())
    logits = np.concatenate(all_logits)
    return 1 / (1 + np.exp(-logits))  # sigmoid -> probability


def train(cfg: dict, sample: int | None = None, out_name: str = "model") -> None:
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")
    tcfg = cfg["transformer"]

    usecols = ["comment_text", "target"] + IDENTITY_COLUMNS + AUX_TOXICITY_COLUMNS
    raw_path = ROOT / cfg["paths"]["raw_train_csv"]
    df = load_raw(raw_path, usecols=usecols)
    df["comment_text"] = df["comment_text"].fillna("")

    if sample:
        df = df.sample(n=min(sample, len(df)), random_state=cfg["seed"]).reset_index(drop=True)
        print(f"[smoke test] subsampled to {len(df)} rows")

    train_df, val_df = train_val_split(df, val_size=cfg["data"]["val_size"], seed=cfg["seed"])
    train_df = prepare_training_frame(train_df, weighting=tcfg["weighting_scheme"])
    val_df = prepare_training_frame(val_df, weighting="none")  # weighting only applies to the training loss

    tokenizer = AutoTokenizer.from_pretrained(tcfg["model_name"])
    train_ds = ToxicityDataset(train_df, tokenizer, tcfg["max_len"], AUX_TOXICITY_COLUMNS)
    val_ds = ToxicityDataset(val_df, tokenizer, tcfg["max_len"], AUX_TOXICITY_COLUMNS)

    train_loader = DataLoader(train_ds, batch_size=tcfg["batch_size"], shuffle=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=tcfg["eval_batch_size"], shuffle=False, num_workers=2)

    model = MultiTaskToxicityModel(
        model_name=tcfg["model_name"],
        n_aux_labels=len(AUX_TOXICITY_COLUMNS),
        use_multitask_heads=tcfg["use_multitask_heads"],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg["weight_decay"])
    total_steps = (len(train_loader) // tcfg["grad_accum_steps"]) * tcfg["epochs"]
    warmup_steps = int(total_steps * tcfg["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = torch.amp.GradScaler(device.type, enabled=tcfg["fp16"] and device.type == "cuda")

    model_dir = ROOT / cfg["paths"]["model_dir"]
    model_dir.mkdir(parents=True, exist_ok=True)

    global_step = 0
    for epoch in range(tcfg["epochs"]):
        model.train()
        t0 = time.time()
        running_loss = 0.0
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            target = batch["target"].to(device)
            weight = batch["sample_weight"].to(device)
            aux_target = batch.get("aux_target")
            aux_target = aux_target.to(device) if aux_target is not None else None

            with torch.autocast(device_type=device.type, enabled=tcfg["fp16"] and device.type == "cuda"):
                main_logit, aux_logits = model(input_ids, attention_mask)
                loss = multitask_loss(
                    main_logit, target, weight, aux_logits, aux_target, tcfg["aux_loss_weight"]
                ) / tcfg["grad_accum_steps"]

            scaler.scale(loss).backward()
            running_loss += loss.item() * tcfg["grad_accum_steps"]

            if (step + 1) % tcfg["grad_accum_steps"] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % tcfg["log_every"] == 0:
                    elapsed = time.time() - t0
                    print(f"epoch {epoch} step {global_step}/{total_steps} "
                          f"loss={running_loss / tcfg['log_every']:.4f} "
                          f"lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.0f}s)")
                    running_loss = 0.0

                if global_step % tcfg["checkpoint_every_steps"] == 0:
                    # Overwrite a single "latest" file rather than one per
                    # step count -- a disconnect only ever needs the most
                    # recent checkpoint, and a multi-hour run checkpointing
                    # every few thousand steps would otherwise write dozens
                    # of full ~370MB state dicts (a real problem on Kaggle,
                    # where /kaggle/working has a bounded output size).
                    ckpt_path = model_dir / f"{out_name}_latest.pt"
                    torch.save(model.state_dict(), ckpt_path)
                    print(f"checkpoint saved ({global_step} steps): {ckpt_path}")

        # End-of-epoch validation with the real bias metric, not just loss.
        val_scores = run_inference(model, val_loader, device)
        val_df_eval = val_df.copy()
        val_df_eval["prediction"] = val_scores
        result = compute_bias_metrics(val_df_eval, label_col="target", pred_col="prediction")
        print(f"\n=== epoch {epoch} validation ===")
        print(result.summary())
        print(result.per_subgroup)

    final_path = model_dir / f"{out_name}_final.pt"
    torch.save(model.state_dict(), final_path)
    print(f"\nSaved final model to {final_path}")

    # Persist the LAST epoch's bias tables + full per-row val predictions --
    # consumed by notebooks/03_error_analysis.ipynb, src/thresholds.py, and
    # the README's result tables. Written under out_name so each ablation
    # run (e.g. deberta_multitask vs deberta_singlehead) gets its own files
    # instead of overwriting the previous run's numbers.
    out_dir = ROOT / cfg["paths"]["reports_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)
    result.per_subgroup.to_csv(out_dir / f"{out_name}_per_subgroup.csv")
    result.summary().to_csv(out_dir / f"{out_name}_summary.csv")
    val_out_cols = ["comment_text", "target", "prediction"] + IDENTITY_COLUMNS
    val_df_eval[val_out_cols].to_csv(out_dir / f"{out_name}_val_predictions.csv", index=False)
    print(f"Saved bias tables + val predictions to {out_dir}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=str, default=str(ROOT / "config" / "config.yaml"))
    parser.add_argument("--sample", type=int, default=None, help="subsample rows for a smoke test")
    parser.add_argument("--out-name", type=str, default="deberta_multitask")
    args = parser.parse_args()

    config = load_config(args.config)
    train(config, sample=args.sample, out_name=args.out_name)


## 3. Kaggle-specific config

Same hyperparameters as `config/config.yaml` in the repo; only the paths differ (point at `/kaggle/input/...` and write outputs under `/kaggle/working/`).

In [ ]:
%%writefile config.yaml
seed: 42

paths:
  raw_train_csv: /kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/train.csv
  raw_test_csv: /kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/test.csv
  processed_dir: /kaggle/working/data/processed
  model_dir: /kaggle/working/models
  reports_dir: /kaggle/working/reports
  figures_dir: /kaggle/working/reports/figures

data:
  val_size: 0.05
  dev_subsample_size: 200000
  min_per_subgroup_dev: 500
  toxic_threshold: 0.5
  identity_threshold: 0.5

identity_columns:
  - male
  - female
  - homosexual_gay_or_lesbian
  - christian
  - jewish
  - muslim
  - black
  - white
  - psychiatric_or_mental_illness

aux_columns:
  - severe_toxicity
  - obscene
  - threat
  - insult
  - identity_attack
  - sexual_explicit

transformer:
  model_name: microsoft/deberta-v3-base
  max_len: 220
  batch_size: 32
  eval_batch_size: 64
  lr: 2.0e-5
  warmup_ratio: 0.05
  epochs: 1   # 1-2 is the spec's range; start with 1 for a faster real
              # result (~3.5-4.5h vs ~7-9h) -- bump to 2 for a rerun if
              # time/GPU-quota allows once this arm's numbers are in.
  weight_decay: 0.01
  fp16: true
  aux_loss_weight: 0.25
  use_multitask_heads: true   # flip to false + rerun for the ablation
  weighting_scheme: metric_derived   # none | inverse_frequency | metric_derived -- flip for the ablation table
  grad_accum_steps: 1
  max_grad_norm: 1.0
  log_every: 100
  checkpoint_every_steps: 3000   # overwrites a single _latest.pt -- see src/train.py


## 4. Smoke test (structural, ~1-2 min)

Runs the full loop end-to-end on 300 rows before committing to a multi-hour run -- catches config/schema mistakes cheaply. Uses `subprocess` + a raised exception (not a bare `!` shell call) specifically so a failure here **halts the notebook** instead of silently falling through to the multi-hour cell below -- a bare `!python ...` returning nonzero does not stop batch execution on its own, which would otherwise burn GPU quota on a broken run.

In [ ]:
import subprocess
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--sample', '300', '--out-name', 'smoketest'])
if r.returncode != 0:
    raise RuntimeError('Smoke test failed (see output above) -- aborting before the full run.')
print('SMOKE TEST PASSED')


## 5. Full training run

This is the multi-hour cell. Metrics (overall AUC + the three bias power-means) print after every epoch; the final state_dict is saved to `/kaggle/working/models/deberta_multitask_final.pt` -- download it from the notebook's Output tab, or add `/kaggle/working` as a new Kaggle Dataset from the notebook's "Save Version" flow so it persists past the session.

In [ ]:
import subprocess
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--out-name', 'deberta_multitask'])
if r.returncode != 0:
    raise RuntimeError('Training run failed (see output above).')
print('TRAINING RUN COMPLETE')


## 6. Copy results back into the local repo

After downloading `/kaggle/working/reports/*.csv` from the Output tab, drop them into `reports/` locally and update the README's result tables + `reports/results.md` log.